# 01 - Emit an external-agent trace

Run the telemetry initialization once during agent startup, before importing any framework that should be auto-instrumented. The sample interaction below uses a manual span so it works without requiring a specific agent framework.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

if not Path('.env').exists():
    raise FileNotFoundError('Copy .env.example to .env and fill in the required values.')

load_dotenv('.env')

required_variables = (
    'APPLICATIONINSIGHTS_CONNECTION_STRING',
    'AGENT_NAME',
    'OTEL_AGENT_ID',
)
missing = [name for name in required_variables if not os.getenv(name)]
if missing:
    raise ValueError(f'Missing required environment variables: {missing}')

os.environ.setdefault('AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING', 'true')
os.environ.setdefault('OTEL_SEMCONV_STABILITY_OPT_IN', 'gen_ai_latest_experimental')

In [ ]:
from microsoft.opentelemetry import use_microsoft_opentelemetry

use_microsoft_opentelemetry(
    enable_azure_monitor=True,
    azure_monitor_connection_string=os.environ['APPLICATIONINSIGHTS_CONNECTION_STRING'],
    sampling_ratio=1.0,
)

print('OpenTelemetry export to Application Insights is configured.')

In [ ]:
from opentelemetry import trace

tracer = trace.get_tracer('external-agent-notebook')

with tracer.start_as_current_span('invoke_agent') as span:
    span.set_attribute('gen_ai.operation.name', 'invoke_agent')
    span.set_attribute('gen_ai.agent.id', os.environ['OTEL_AGENT_ID'])
    span.set_attribute('gen_ai.agent.name', os.environ['AGENT_NAME'])

    # Replace this block with one invocation of your externally hosted agent.
    user_request = 'What is the weather in Seattle?'
    agent_response = 'Sample response emitted by the notebook trace.'

    # Do not attach prompt or response content unless your data policy permits it.
    span.set_attribute('external_agent.request_length', len(user_request))
    span.set_attribute('external_agent.response_length', len(agent_response))

provider = trace.get_tracer_provider()
if hasattr(provider, 'force_flush'):
    flushed = provider.force_flush(timeout_millis=30000)
    if flushed is False:
        raise RuntimeError('The OpenTelemetry provider did not flush within 30 seconds.')

print(f"Trace emitted with gen_ai.agent.id={os.environ['OTEL_AGENT_ID']}")
print('Application Insights ingestion and Foundry display can take 2-5 minutes.')

## Framework integration

For LangChain or LangGraph, install `microsoft-opentelemetry[langchain]` and enable the `langchain` instrumentation option with the same agent ID. For another framework, preserve the `gen_ai.agent.id` attribute on every agent span.